# FRED-MD Final

Huber loss (1.35), no intercept, $\Delta=n/20$, and at most four changes. Unpenalized: **10 folds**; L1: **2 folds**.

In [1]:
from pathlib import Path
import os, json, pickle
import numpy as np
import pandas as pd
from IPython.display import display
if not Path('WERM').is_dir() and Path('../WERM').is_dir():
    os.chdir('..')
RUN_EXPERIMENTS = False
with open('FRED-MD data/fredmd_data.pkl', 'rb') as handle:
    data = pickle.load(handle)
X = np.asarray(data['X'], dtype=float)
y = np.asarray(data['y'], dtype=float).reshape(-1)
dates = np.asarray(data['date']).astype(str)
assert X.shape == (773, 114)
assert dates[[0, -1]].tolist() == ['1960-01', '2024-05']
results = json.loads(Path('results/FRED-MD_paper.json').read_text())


## Unpenalized Huber: 10-fold cross-validation

In [2]:
if RUN_EXPERIMENTS:
    import jax.numpy as jnp
    from WERM.experiment_utils import weighted_erm_joint_cv
    def huber(prediction, truth, epsilon=1.35):
        residual = prediction - truth
        absolute = jnp.abs(residual)
        return jnp.where(absolute <= epsilon, 0.5 * residual**2,
                         epsilon * (absolute - 0.5 * epsilon))
    def absolute(prediction, truth):
        return jnp.abs(prediction - truth)
    selected, cp, diagnostics = weighted_erm_joint_cv(
        X=X, y=y, M=huber, M_scorer=absolute, Lmax=5,
        delta_fraction=1/20, l1_constants=(0.0,), num_folds=10,
        n_jobs=4, standardize_y=False, selection_strategy='exhaustive',
        return_diagnostics=True, penalty_norm='none')
    results['unpenalized'] = {
        'num_folds': 10, 'selected_change_points': int(selected[0])-1,
        'estimated_change_points': np.asarray(cp, dtype=int).tolist(),
        'score_curve': [{'change_points': int(candidate[0])-1,
                         'cv_mean_absolute_error': float(score)/len(y)}
                        for candidate, score in zip(diagnostics['candidates'], diagnostics['scores'])]}


In [3]:
unpenalized = results['unpenalized']
assert unpenalized['num_folds'] == 10
assert unpenalized['selected_change_points'] == 0
assert unpenalized['estimated_change_points'] == []
curve = pd.DataFrame(unpenalized['score_curve'])[['change_points', 'cv_mean_absolute_error']]
assert int(curve.loc[curve.cv_mean_absolute_error.idxmin(), 'change_points']) == 0
display(curve.rename(columns={'change_points': 'Candidate changes',
                             'cv_mean_absolute_error': '10-fold CV mean absolute error'}))
print('10-fold unpenalized CV selects zero change points.')


,Candidate changes,10-fold CV mean absolute error
0,0,0.544114
1,1,0.601205
2,2,0.692216
3,3,0.780620
4,4,0.796762


10-fold unpenalized CV selects zero change points.


## Huber + L1: two-fold full joint CV

In [4]:
if RUN_EXPERIMENTS:
    selected, cp, diagnostics = weighted_erm_joint_cv(
        X=X, y=y, M=huber, M_scorer=absolute, Lmax=5,
        delta_fraction=1/20, l1_constants=(0.03, 0.30, 3.00), num_folds=2,
        n_jobs=4, standardize_y=False, selection_strategy='exhaustive',
        return_diagnostics=True, penalty_norm='l1', segment_counts=(1,2,3,4,5))
    results['penalized'] = {'num_folds': 2, 'C': float(selected[1]),
        'selected_change_points': len(cp),
        'change_point_dates': dates[np.asarray(cp, dtype=int)].tolist()}
    Path('results/FRED-MD_paper.json').write_text(json.dumps(results, indent=2)+'\n')
penalized = results['penalized']
assert penalized['num_folds'] == 2
assert penalized['change_point_dates'] == ['1980-09', '2009-06']
display(pd.DataFrame([{'Method': 'Huber + L1', 'CV folds': 2,
    'C': penalized['C'], 'Changes': penalized['selected_change_points'],
    'Dates': '; '.join(penalized['change_point_dates'])}]))


,Method,CV folds,C,Changes,Dates
0,Huber + L1,2,0.3,2,1980-09; 2009-06
